# Network Construction — Hypotheses 1–2

Constructs a weighted organization co-occurrence network from smart-city news articles.


In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter

# 1. Load the news dataset
# Organization names are retained in Korean because they are source-data entities.
df = pd.read_csv("smart_city_news_articles.csv")

# 2. Preprocess organization lists and generate pairwise co-occurrence edges
edge_list = []
for _, row in df.iterrows():
    organizations = [
        str(x).strip()
        for x in str(row["organizations"]).split(",")
        if str(x).strip()
    ]
    organization_pairs = list(combinations(sorted(set(organizations)), 2))
    edge_list.extend(organization_pairs)

# 3. Count co-occurrences and remove one-off edges (weight >= 2)
edge_counts = Counter(edge_list)
filtered_edges = [
    (source, target, weight)
    for (source, target), weight in edge_counts.items()
    if weight >= 2
]

# 4. Build an undirected weighted graph
G = nx.Graph()
G.add_weighted_edges_from(filtered_edges)

# 5. Create the node table
nodes_df = pd.DataFrame(list(G.nodes()), columns=["Label"])
nodes_df.insert(0, "Id", range(1, len(nodes_df) + 1))

# 6. Map organization labels to numeric node IDs
node_id_map = dict(zip(nodes_df["Label"], nodes_df["Id"]))

# 7. Create the edge table
edges_data = []
for source, target, data in G.edges(data=True):
    edges_data.append({
        "Source": node_id_map[source],
        "Target": node_id_map[target],
        "Weight": data["weight"]
    })
edges_df = pd.DataFrame(edges_data)

# 8. Save graph tables for downstream analysis and Gephi
nodes_df.to_csv("nodes.csv", index=False, encoding="utf-8-sig")
edges_df.to_csv("edges.csv", index=False, encoding="utf-8-sig")
